# 🔬 Autonomous AI Researcher — Ablation Study (Kaggle)

**Qwen2.5-14B chạy LOCAL trên Kaggle GPU — load 1 lần, dùng cho cả 6 experiments.**

| Exp | Cấu hình | Mục đích |
|-----|----------|----------|
| 1 | Vanilla LLM (zero-shot) | Baseline thấp nhất |
| 2 | RAG truyền thống | Baseline có truy hồi |
| 3 | Agent (chưa fine-tune) | Đo lợi ích agentic flow |
| 4 | Agent + Reviewer LoRA | Đo đóng góp Reviewer FT |
| 5 | Agent + Reranker FT | Đo đóng góp Reranker FT |
| 6 | Agent đầy đủ (cả hai) | Hệ thống hoàn chỉnh |

**Kaggle Inputs cần add (sidebar phải → Add Data):**
1. `autonomous-researcher-code` — `src_code.zip`
2. `qwen25-14b-instruct` — Qwen2.5-14B model weights (backbone cho cả 4 agent)
3. `reviewer-lora-adapter` — `reviewer_lora_v2_adapter/` (cho Exp4, Exp6)
4. `reranker-ft-model` — `reranker_ft/` (cho Exp5, Exp6)

**Lưu ý VRAM:** Qwen2.5-14B @ 4-bit ≈ 9GB. Kaggle T4 = 15GB. Chỉ load model 1 lần duy nhất.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: Cài đặt thư viện
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

pkgs = [
    "langgraph",
    "sentence-transformers",
    "chromadb",
    "trafilatura",
    "duckduckgo-search",
    "openai",          # chỉ dùng cho LLM-as-Judge (Groq) ở bước metrics
    "peft",
    "bitsandbytes",
    "accelerate",
    "PyYAML",
    "pandas",
    "matplotlib",
    "python-dotenv",
]

print("📦 Cài đặt thư viện...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("✅ Xong!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: Giải nén source code và setup working directory
# ═══════════════════════════════════════════════════════════════
import os, sys, shutil, zipfile

WORKING_DIR = "/kaggle/working"

POSSIBLE_ZIP_PATHS = [
    "/kaggle/input/autonomous-researcher-code/src_code.zip",
    "/kaggle/input/autonomous-researcher/src_code.zip",
    "/kaggle/input/autonomous-researcher-code/autonomous_researcher_kaggle.zip",
]

zip_path = None
for p in POSSIBLE_ZIP_PATHS:
    if os.path.exists(p):
        zip_path = p
        break

if zip_path:
    print(f"📦 Tìm thấy zip: {zip_path}")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(WORKING_DIR)
    print(f"✅ Đã giải nén vào {WORKING_DIR}")
else:
    for base in ["/kaggle/input/autonomous-researcher-code", "/kaggle/input/autonomous-researcher"]:
        if os.path.isdir(base) and os.path.exists(os.path.join(base, "src")):
            print(f"📂 Copy từ: {base}")
            for item in os.listdir(base):
                src = os.path.join(base, item)
                dst = os.path.join(WORKING_DIR, item)
                if os.path.isdir(src):
                    shutil.copytree(src, dst, dirs_exist_ok=True)
                else:
                    shutil.copy2(src, dst)
            zip_path = base
            break

if not zip_path:
    raise FileNotFoundError("❌ Không tìm thấy source code! Add dataset src_code.zip vào Kaggle Input.")

os.chdir(WORKING_DIR)
sys.path.insert(0, WORKING_DIR)

# Tạo thư mục cần thiết
for d in ["logs/traces/ablation", "data/raw_outputs", "data/benchmarks", "logs/experiments"]:
    os.makedirs(d, exist_ok=True)

print(f"📍 Working dir: {os.getcwd()}")
print("📁 Top-level:", sorted([f for f in os.listdir(WORKING_DIR) if not f.startswith('.')]))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: Load Qwen2.5-14B MỘT LẦN DUY NHẤT vào RAM/VRAM
# (4-bit NF4 quantization: 28GB → ~9GB VRAM)
# ═══════════════════════════════════════════════════════════════
import sys, os
sys.path.insert(0, "/kaggle/working")
from dotenv import load_dotenv
load_dotenv() # Load env vars early

from src.models.llm_server import LLMClient

print(f"🚀 Loading Qwen2.5-14B from: {QWEN_MODEL_PATH}")
print("⏳ Ước tính: 1-2 phút để load + quantize...")

SHARED_LLM = LLMClient(
    model_name=QWEN_MODEL_PATH,
    backend="transformers",
    quantization="4bit",   # NF4 double quant: 28GB → ~9GB VRAM
    temperature=0.7,
    max_tokens=4096,
)

print("\n✅ Model đã load xong vào VRAM!")
print("   SHARED_LLM sẽ được dùng chung cho TẤT CẢ 6 experiments.")
print("   Không load lại — tiết kiệm thời gian và VRAM.")

# Kiểm tra GPU memory
try:
    import torch
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved  = torch.cuda.memory_reserved()  / 1024**3
    print(f"\n   GPU VRAM used: {allocated:.1f} GB allocated, {reserved:.1f} GB reserved")
except:
    pass

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: Load Qwen2.5-14B MỘT LẦN DUY NHẤT vào RAM/VRAM
# (4-bit NF4 quantization: 28GB → ~9GB VRAM)
# ═══════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, "/kaggle/working")
from src.models.llm_server import LLMClient

print(f"🚀 Loading Qwen2.5-14B from: {QWEN_MODEL_PATH}")
print("⏳ Ước tính: 1-2 phút để load + quantize...")

SHARED_LLM = LLMClient(
    model_name=QWEN_MODEL_PATH,
    backend="transformers",
    quantization="4bit",   # NF4 double quant: 28GB → ~9GB VRAM
    temperature=0.7,
    max_tokens=4096,
)

print("\n✅ Model đã load xong vào VRAM!")
print("   SHARED_LLM sẽ được dùng chung cho TẤT CẢ 6 experiments.")
print("   Không load lại — tiết kiệm thời gian và VRAM.")

# Kiểm tra GPU memory
try:
    import torch
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved  = torch.cuda.memory_reserved()  / 1024**3
    print(f"\n   GPU VRAM used: {allocated:.1f} GB allocated, {reserved:.1f} GB reserved")
except:
    pass

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: Patch YAML configs với đường dẫn Kaggle thực tế
# ═══════════════════════════════════════════════════════════════
import yaml, os

def patch_config(config_path, **overrides):
    """Override nested YAML keys dạng 'a.b.c' = value."""
    if not os.path.exists(config_path):
        print(f"⚠ Không tìm thấy: {config_path}")
        return
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    for dotkey, val in overrides.items():
        keys = dotkey.split(".")
        node = cfg
        for k in keys[:-1]:
            node = node.setdefault(k, {})
        node[keys[-1]] = val
    with open(config_path, "w", encoding="utf-8") as f:
        yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
    print(f"  ✅ {os.path.basename(config_path)}")

print("Patching experiment configs...")

# Exp1–3: không có fine-tuned components
# (llm config trỏ về Groq trong YAML nhưng SHARED_LLM override hết → không cần patch)

# Exp4: Reviewer LoRA
patch_config(
    "configs/experiments/exp4_agent_ft_reviewer.yaml",
    **{
        "fine_tuned.reviewer.enabled":     bool(REVIEWER_ADAPTER_PATH),
        "fine_tuned.reviewer.base_model":  QWEN_MODEL_PATH,
        "fine_tuned.reviewer.adapter_path": REVIEWER_ADAPTER_PATH or "reviewer_lora_v2_adapter",
    }
)

# Exp5: Reranker
patch_config(
    "configs/experiments/exp5_agent_ft_reranker.yaml",
    **{
        "fine_tuned.reranker.enabled":    bool(RERANKER_PATH),
        "fine_tuned.reranker.model_path": RERANKER_PATH or "reranker_ft",
    }
)

# Exp6: Cả hai
patch_config(
    "configs/experiments/exp6_agent_full.yaml",
    **{
        "fine_tuned.reviewer.enabled":     bool(REVIEWER_ADAPTER_PATH),
        "fine_tuned.reviewer.base_model":  QWEN_MODEL_PATH,
        "fine_tuned.reviewer.adapter_path": REVIEWER_ADAPTER_PATH or "reviewer_lora_v2_adapter",
        "fine_tuned.reranker.enabled":    bool(RERANKER_PATH),
        "fine_tuned.reranker.model_path": RERANKER_PATH or "reranker_ft",
    }
)

print("\n✅ Tất cả configs đã được cập nhật")
if not REVIEWER_ADAPTER_PATH:
    print("⚠ Exp4/6: reviewer FT disabled (không tìm thấy adapter)")
if not RERANKER_PATH:
    print("⚠ Exp5/6: reranker FT disabled (không tìm thấy model)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7: Tính metrics từ raw JSONL
# LLM-as-Judge dùng Groq (nhẹ, không cần GPU) có xoay key
# ═══════════════════════════════════════════════════════════════
import json, os
import pandas as pd
from dotenv import load_dotenv
from src.evaluation.metrics import (
    answer_f1, citation_precision, ndcg_at_k, mrr,
    is_chunk_relevant, is_evidence_sufficient, reviewer_accuracy, reviewer_f1
)
from src.evaluation.judge import LLMJudge
from src.models.llm_server import LLMClient
import src.models.llm_server

# 1. Load keys từ .env hoặc paste trực tiếp
load_dotenv(override=True)

# Nếu bạn muốn điền trực tiếp chuỗi 20 keys, dán vào biến MY_KEYS dưới đây:
MY_KEYS = "" # Ví dụ: "gsk_key1,gsk_key2,gsk_key3..."

if MY_KEYS:
    os.environ["GROQ_API_KEYS"] = MY_KEYS
    os.environ["GROQ_API_KEY"] = MY_KEYS.split(",")[0].strip()
elif os.getenv("GROQ_API_KEYS"):
    # Sử dụng các keys đã load từ file .env
    os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEYS").split(",")[0].strip()
else:
    # Fallback key duy nhất
    fallback = "YOUR_GROQ_API_KEY_HERE"
    os.environ["GROQ_API_KEYS"] = fallback
    os.environ["GROQ_API_KEY"] = fallback

# Reset KeyRotator singleton để nhận diện list keys mới
src.models.llm_server._key_rotator = None

# 2. Khởi tạo LLM Client cho Judge (sẽ tự động xoay key khi dính Rate Limit 429)
judge_client = LLMClient(
    model_name="llama-3.3-70b-versatile",
    backend="openai",
    api_base="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
)
judge = LLMJudge(llm_client=judge_client)

RUBRIC = (
    "You are a strict academic reviewer. Score from 1.0 to 5.0:\n"
    "COMPREHENSIVENESS: 1.0=1-3 sentences; 3.0=basic paragraphs; 5.0=multi-section report with tables.\n"
    "DEPTH: 1.0=generic/fluffy; 3.0=mentions facts but shallow; 5.0=precise metrics, authors, mechanics."
)

questions_map = {q.id: q for q in questions}
raw_dir = "data/raw_outputs"
all_results = []

for jsonl_file in sorted(os.listdir(raw_dir)):
    if not jsonl_file.endswith(".jsonl"):
        continue
    exp_name = jsonl_file.replace("_raw.jsonl", "")
    print(f"\n📊 {exp_name}")

    records = [json.loads(l) for l in open(os.path.join(raw_dir, jsonl_file))]

    for rec in records:
        q_id    = rec["question_id"]
        q_obj   = questions_map.get(q_id)
        supporting_facts = q_obj.supporting_facts if q_obj else []
        report  = rec.get("final_report", "")
        gt      = rec.get("ground_truth_answer", "")
        cits    = rec.get("citations", [])
        evs     = rec.get("collected_evidence", [])
        success = rec.get("success", False)
        decisions = rec.get("reviewer_decisions", [])

        f1    = answer_f1(report, gt) if report else 0.0
        cit_p = 0.0
        try:
            cit_p = citation_precision(report_body=report, citations=cits,
                                       collected_evidence=evs, judge=judge) or 0.0
        except Exception as e:
            print(f"  ⚠ citation_precision: {e}")

        # Tính ndcg và mrr
        ndcg_val = 0.0
        mrr_val = 0.0
        if supporting_facts:
            sorted_evidence = sorted(evs, key=lambda x: x.get("score", 0.0), reverse=True)
            relevance_scores = [is_chunk_relevant(ev.get("text", ""), supporting_facts) for ev in sorted_evidence]
            ndcg_val = ndcg_at_k(relevance_scores, k=5)
            mrr_val = mrr(relevance_scores)

        # Tính reviewer accuracy và F1
        rev_acc = 1.0
        rev_f1 = 1.0
        if decisions and supporting_facts:
            reviewer_preds = [d["sufficient"] for d in decisions]
            reviewer_gts = [is_evidence_sufficient(d["collected_evidence"], supporting_facts) for d in decisions]
            rev_acc = reviewer_accuracy(reviewer_preds, reviewer_gts)
            rev_f1 = reviewer_f1(reviewer_preds, reviewer_gts)

        comp = depth = overall = 1.0
        if report and success:
            try:
                jr      = judge.judge_report(report, rec["question"], RUBRIC)
                comp    = float(jr.comprehensiveness)
                depth   = float(jr.depth_of_research)
                overall = float(jr.overall_score)
            except Exception as e:
                print(f"  ⚠ judge: {e}")

        all_results.append({
            "experiment":              exp_name,
            "question_id":             q_id,
            "answer_f1":               round(f1, 4),
            "citation_precision":      round(cit_p, 4),
            "judge_comprehensiveness": round(comp, 3),
            "judge_depth":             round(depth, 3),
            "judge_overall":           round(overall, 3),
            "ndcg_at_5":               round(ndcg_val, 4),
            "mrr":                     round(mrr_val, 4),
            "reviewer_accuracy":        round(rev_acc, 4),
            "reviewer_f1":              round(rev_f1, 4),
            "step_count":              rec.get("step_count", 0),
            "success":                 int(success),
        })
        print(f"  {q_id}: F1={f1:.3f} | CiteP={cit_p:.3f} | Judge={overall:.2f} | NDCG@5={ndcg_val:.3f} | RevAcc={rev_acc:.3f} | ✓={success}")

df_all = pd.DataFrame(all_results)
df_all.to_csv("data/raw_outputs/all_metrics_detail.csv", index=False)
print("\n✅ Saved: data/raw_outputs/all_metrics_detail.csv")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8: Bảng so sánh Ablation + Biểu đồ
# ═══════════════════════════════════════════════════════════════
import pandas as pd, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

df_all = pd.read_csv("data/raw_outputs/all_metrics_detail.csv")

agg = df_all.groupby("experiment").agg(
    task_success_rate=("success", "mean"),
    answer_f1=("answer_f1", "mean"),
    citation_precision=("citation_precision", "mean"),
    judge_comprehensiveness=("judge_comprehensiveness", "mean"),
    judge_depth=("judge_depth", "mean"),
    judge_overall=("judge_overall", "mean"),
    ndcg_at_5=("ndcg_at_5", "mean"),
    mrr=("mrr", "mean"),
    reviewer_accuracy=("reviewer_accuracy", "mean"),
    reviewer_f1=("reviewer_f1", "mean"),
    avg_steps=("step_count", "mean"),
    n_questions=("question_id", "count"),
).round(4).reset_index()

EXP_LABELS = {
    "exp1_vanilla_llm":       "Exp1: Vanilla LLM",
    "exp2_rag_baseline":      "Exp2: RAG",
    "exp3_agent_base":        "Exp3: Agent",
    "exp4_agent_ft_reviewer": "Exp4: +Reviewer FT",
    "exp5_agent_ft_reranker": "Exp5: +Reranker FT",
    "exp6_agent_full":        "Exp6: Full System",
}
order = list(EXP_LABELS.keys())
agg["Config"]   = agg["experiment"].map(EXP_LABELS).fillna(agg["experiment"])
agg["sort_key"] = agg["experiment"].map({k: i for i, k in enumerate(order)}).fillna(99)
agg = agg.sort_values("sort_key").drop(columns=["sort_key", "experiment"])

cols = ["Config", "task_success_rate", "answer_f1", "citation_precision",
        "judge_comprehensiveness", "judge_depth", "judge_overall",
        "ndcg_at_5", "mrr", "reviewer_accuracy", "reviewer_f1", "avg_steps"]
agg = agg[[c for c in cols if c in agg.columns]]

agg.to_csv("logs/experiments/ablation_comparison.csv", index=False)
agg.to_json("logs/experiments/ablation_comparison.json", orient="records", indent=2)

print("🏆 ABLATION STUDY — Autonomous AI Researcher (Qwen2.5-14B local)")
print("=" * 110)
print(agg.to_string(index=False))
print("=" * 110)

# ── Bar charts ──────────────────────────────────────────────
short = ["Vanilla", "RAG", "Agent", "+Rev.FT", "+Rk.FT", "Full"]
labels = short[:len(agg)]
metrics_plot = [
    ("answer_f1",           "Answer F1",             "#4C9BE8"),
    ("citation_precision",  "Citation Precision",    "#F4845F"),
    ("judge_overall",       "LLM-Judge Overall",     "#6BCB77"),
    ("task_success_rate",   "Task Success Rate",     "#FFD93D"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Ablation Study — Autonomous AI Researcher (Qwen2.5-14B)",
             fontsize=15, fontweight="bold")

for ax, (col, title, color) in zip(axes.flat, metrics_plot):
    if col not in agg.columns:
        ax.set_visible(False); continue
    vals = agg[col].fillna(0).tolist()
    bars = ax.bar(labels[:len(vals)], vals, color=color, edgecolor="white", linewidth=1.2)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylim(0, max(vals) * 1.3 if max(vals) > 0 else 1.0)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("logs/experiments/ablation_chart.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n📄 logs/experiments/ablation_comparison.csv")
print("📊 logs/experiments/ablation_chart.png")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8: Bảng so sánh Ablation + Biểu đồ
# ═══════════════════════════════════════════════════════════════
import pandas as pd, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

df_all = pd.read_csv("data/raw_outputs/all_metrics_detail.csv")

agg = df_all.groupby("experiment").agg(
    task_success_rate=("success", "mean"),
    answer_f1=("answer_f1", "mean"),
    citation_precision=("citation_precision", "mean"),
    judge_comprehensiveness=("judge_comprehensiveness", "mean"),
    judge_depth=("judge_depth", "mean"),
    judge_overall=("judge_overall", "mean"),
    avg_steps=("step_count", "mean"),
    n_questions=("question_id", "count"),
).round(4).reset_index()

EXP_LABELS = {
    "exp1_vanilla_llm":       "Exp1: Vanilla LLM",
    "exp2_rag_baseline":      "Exp2: RAG",
    "exp3_agent_base":        "Exp3: Agent",
    "exp4_agent_ft_reviewer": "Exp4: +Reviewer FT",
    "exp5_agent_ft_reranker": "Exp5: +Reranker FT",
    "exp6_agent_full":        "Exp6: Full System",
}
order = list(EXP_LABELS.keys())
agg["Config"]   = agg["experiment"].map(EXP_LABELS).fillna(agg["experiment"])
agg["sort_key"] = agg["experiment"].map({k: i for i, k in enumerate(order)}).fillna(99)
agg = agg.sort_values("sort_key").drop(columns=["sort_key", "experiment"])

cols = ["Config", "task_success_rate", "answer_f1", "citation_precision",
        "judge_comprehensiveness", "judge_depth", "judge_overall", "avg_steps"]
agg = agg[[c for c in cols if c in agg.columns]]

agg.to_csv("logs/experiments/ablation_comparison.csv", index=False)
agg.to_json("logs/experiments/ablation_comparison.json", orient="records", indent=2)

print("🏆 ABLATION STUDY — Autonomous AI Researcher (Qwen2.5-14B local)")
print("=" * 110)
print(agg.to_string(index=False))
print("=" * 110)

# ── Bar charts ──────────────────────────────────────────────
short = ["Vanilla", "RAG", "Agent", "+Rev.FT", "+Rk.FT", "Full"]
labels = short[:len(agg)]
metrics_plot = [
    ("answer_f1",           "Answer F1",             "#4C9BE8"),
    ("citation_precision",  "Citation Precision",    "#F4845F"),
    ("judge_overall",       "LLM-Judge Overall",     "#6BCB77"),
    ("task_success_rate",   "Task Success Rate",     "#FFD93D"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Ablation Study — Autonomous AI Researcher (Qwen2.5-14B)",
             fontsize=15, fontweight="bold")

for ax, (col, title, color) in zip(axes.flat, metrics_plot):
    if col not in agg.columns:
        ax.set_visible(False); continue
    vals = agg[col].fillna(0).tolist()
    bars = ax.bar(labels[:len(vals)], vals, color=color, edgecolor="white", linewidth=1.2)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylim(0, max(vals) * 1.3 if max(vals) > 0 else 1.0)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("logs/experiments/ablation_chart.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n📄 logs/experiments/ablation_comparison.csv")
print("📊 logs/experiments/ablation_chart.png")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9: (OPTIONAL) Chạy lại 1 experiment cụ thể
# ═══════════════════════════════════════════════════════════════
from scripts.generate_raw_data import run_experiment
import os

# Bỏ comment để chạy lại một experiment cụ thể:
# run_experiment(
#     config_path="configs/experiments/exp3_agent_base.yaml",
#     questions=questions,
#     output_dir="data/raw_outputs",
#     shared_llm_client=SHARED_LLM,
# )

print("📁 Output files:")
for f in sorted(os.listdir("data/raw_outputs")):
    p = os.path.join("data/raw_outputs", f)
    n = sum(1 for _ in open(p)) if f.endswith(".jsonl") else "-"
    print(f"  {f} ({n} records)")